In [ ]:
# | default_exp report.generator

# Report Generator
> Generate comprehensive SEO reports with page analysis, duplicate detection, and issue tracking.

In [ ]:
# | export
from typing import Callable
from concurrent.futures import ThreadPoolExecutor, as_completed
import httpx
from sqlmodel import Session
from seootter.article import get_articles_by_website, Article
from seootter.parser.metadata import remove_metadata, extract_html_metadata
from seootter.parser.utils import fetch_url_as_markdown
from seootter.parser.extractors import extract_headers
from seootter.gsc.queries import get_top_queries
from seootter.gsc_client import get_date_range
from seootter.insights.trends import detect_query_trends
from seootter.insights.intent import classify_page_intents
from seootter.insights.keywords import (
    all_missing_queries,
    filter_missing_queries,
    find_green_keywords,
)
from seootter.report.analysis import (
    analyze_article,
    _collect_metadata_field,
    find_duplicate_metadata,
)

FETCH = "::fetch::"


In [ ]:
# | export

# | export


def _collect_page_issues(report: dict, article: Article) -> list[str]:
    issues = []
    r = report.get("title_check", {})
    if r.get("status") == "too_short":
        issues.append(f"Title is too short ({r['length']} chars, min {r['min']})")
    elif r.get("status") == "too_long":
        issues.append(f"Title is too long ({r['length']} chars, max {r['max']})")

    r = report.get("description_check", {})
    if r.get("status") == "missing":
        issues.append("Meta description is missing")
    elif r.get("status") == "too_short":
        issues.append(
            f"Meta description is too short ({r['length']} chars, min {r['min']})"
        )
    elif r.get("status") == "too_long":
        issues.append(
            f"Meta description is too long ({r['length']} chars, max {r['max']})"
        )

    r = report.get("content_check", {})
    if not r.get("is_sufficient"):
        issues.append(f"Content too short ({r.get('word_count', 0)} words, need 300)")

    r = report.get("h1_check", {})
    if r.get("h1_count", 1) == 0:
        issues.append("No H1 heading found")
    elif r.get("h1_count", 1) > 1:
        issues.append(f"Multiple H1 headings ({r['h1_count']})")

    missing = report.get("missing_alts", [])
    if missing:
        issues.append(f"{len(missing)} image(s) missing alt text")

    link_analysis = report.get("link_analysis", {})
    if link_analysis.get("broken_links"):
        issues.append(f"{len(link_analysis['broken_links'])} broken link(s)")

    trends = report.get("trends", {})
    rising = trends.get("rising", [])
    declining = trends.get("declining", [])
    if rising:
        qs = ", ".join(f"{t['query']} ({t['change']}%)" for t in rising[:3])
        issues.append(f"📈 {len(rising)} rising queries: {qs}")
    if declining:
        qs = ", ".join(f"{t['query']} ({t['change']}%)" for t in declining[:3])
        issues.append(f"📉 {len(declining)} declining queries: {qs}")

    return issues


def generate_seo_report(
    session: Session,
    website_id: int,
    domain: str,
    is_quarto: bool,
    title_is_h1: bool = False,
    desc_field: str = "description",
    title_field: str = "title",
    date_field: str = "date",
    fetch_base_url: str | None = None,
    include_insights: bool = True,
    days: int = 30,
    query_limit: int = 500,
    progress_callback: Callable[[int, int, str], None] | None = None,
) -> dict:
    "Generate a comprehensive SEO report for a website."
    articles = get_articles_by_website(session, website_id)
    total = len(articles)
    if progress_callback:
        progress_callback(0, total, f"Found {total} articles")

    # Pre-fetch all FETCH articles in parallel
    fetch_articles = [a for a in articles if a.file_path == FETCH]
    prefetched: dict[str, dict] = {}
    if fetch_articles:

        def _fetch_one(article: Article) -> tuple[str, dict | None]:
            try:
                fetch_url = (
                    article.url.replace(f"https://{domain}", fetch_base_url)
                    if fetch_base_url
                    else article.url
                )
                html = httpx.get(fetch_url, timeout=15).text
                metadata = extract_html_metadata(html)
                content = fetch_url_as_markdown(fetch_url)
                headers = extract_headers(content)
                from trafilatura import fetch_url as tfetch, extract as textract

                thtml = tfetch(article.url)
                trafilatura_content = (
                    textract(thtml, output_format="markdown", favor_recall=True)
                    if thtml
                    else ""
                )
                return article.url, {
                    "metadata": metadata,
                    "content": content,
                    "headers": headers,
                    "trafilatura_content": trafilatura_content,
                }
            except Exception:
                return article.url, None

        with ThreadPoolExecutor(max_workers=10) as pool:
            futures = {pool.submit(_fetch_one, a): a for a in fetch_articles}
            for future in as_completed(futures):
                url, result = future.result()
                if result:
                    prefetched[url] = result

    if progress_callback:
        progress_callback(0, total, "Prefetch complete, analyzing articles...")

    start, end = get_date_range("last_days", days=days)
    issues = {}
    page_reports = []
    article_idx = 0

    for article in articles:
        try:
            report = analyze_article(
                article,
                domain,
                is_quarto,
                title_is_h1=title_is_h1,
                desc_field=desc_field,
                title_field=title_field,
                date_field=date_field,
                fetch_base_url=fetch_base_url,
                prefetched=prefetched,
            )
            if not report or report.get("error"):
                print(f"SKIPPED: {article.url} — {report}")
                article_idx += 1
                continue


            queries = get_top_queries(
                session=session,
                site_url=f"sc-domain:{domain}",
                start_date=start,
                end_date=end,
                page_path=article.url.removeprefix(f"https://{domain}/"),
                limit=500,
                sort_by="impressions",
            )
            if article.file_path == FETCH:
                content = (
                    prefetched[article.url].get("trafilatura_content", "")
                    if article.url in prefetched
                    else ""
                )
            else:
                try:
                    raw = open(article.file_path).read()
                except FileNotFoundError:
                    article_idx += 1
                    continue
                content = remove_metadata(raw)

            all_mq = all_missing_queries(queries=queries, content=content)
            all_mq_string = [q["query"] for q in all_mq]
            # missing_queries = filter_missing_queries(
            #     queries=all_mq_string,
            #     content=content,
            #     focus_keyword=article.focus_keyword or "",
            #     secondary_keywords=article.secondary_keywords or [],
            # )
            missing_queries = all_mq

            page_issues = _collect_page_issues(report, article)

            if include_insights:
                page_path = article.url.removeprefix(f"https://{domain}/")
                insight_start, insight_end = get_date_range("last_days", days=days)
                trends = detect_query_trends(
                    session,
                    f"sc-domain:{domain}",
                    page_path=page_path,
                    days=days,
                    limit=query_limit,
                )
                report["query_intents"] = classify_page_intents(
                    session,
                    f"sc-domain:{domain}",
                    insight_start,
                    insight_end,
                    page_path=page_path,
                    limit=query_limit,
                )
                report["trends"] = {
                    "rising": [t for t in trends if t["trend"] == "rising"],
                    "declining": [t for t in trends if t["trend"] == "declining"],
                }
                report["green_keywords"] = find_green_keywords(
                    session, f"sc-domain:{domain}", page_path, content, days=days
                )

            if page_issues or missing_queries:
                issues[article.url or article.file_path] = {
                    "file_path": article.file_path,
                    "issues": page_issues,
                    "data": {
                        "missing_queries": missing_queries,
                        "keyword_placement": report.get("keyword_placement", {}),
                    },
                }
            page_reports.append(report)
        except Exception as e:
            print(f"Error analyzing {article.file_path}: {e}")

        article_idx += 1
        if progress_callback:
            progress_callback(
                article_idx,
                total,
                f"Analyzed {article_idx}/{total}: {article.url or article.file_path}",
            )

    if progress_callback:
        progress_callback(total, total, "Checking for duplicates...")
    duplicate_titles = find_duplicate_metadata(session, "title", website_id)
    duplicate_descriptions = find_duplicate_metadata(session, "description", website_id)

    if progress_callback:
        progress_callback(total, total, "Done")

    return {
        "total_pages": len(articles),
        "pages_analyzed": len(page_reports),
        "page_reports": page_reports,
        "issues": issues,
        "summary": {
            "pages_with_issues": len(issues),
            "total_issues": sum(len(v["issues"]) for v in issues.values()),
            "duplicate_titles_count": len(duplicate_titles),
            "duplicate_descriptions_count": len(duplicate_descriptions),
        },
    }


In [ ]:
# | export


def print_issues_report(report: dict) -> None:
    "Print comprehensive SEO report with verbose analysis."
    try:
        issues, summary = report["issues"], report["summary"]

        # Header
        print("=" * 70)
        print("SEO ANALYSIS REPORT")
        print("=" * 70)
        print(
            f"Pages analysed   : {report['pages_analyzed']} / {report['total_pages']}"
        )
        print(f"Pages with issues: {summary['pages_with_issues']}")
        print(f"Total issue count: {summary['total_issues']}")
        print(f"Duplicate titles : {summary['duplicate_titles_count']}")
        print(f"Duplicate descs  : {summary['duplicate_descriptions_count']}")
        print("=" * 70)

        # Sort by issue count
        for url, data in sorted(
            issues.items(), key=lambda x: len(x[1]["issues"]), reverse=True
        ):
            count = len(data["issues"])
            label = f"{count} issue{'s' if count != 1 else ''}"
            print(f"\n{'=' * 70}")
            print(f"[{label}] {url}")
            print(f"File: {data['file_path']}")
            print("-" * 70)

            # Issues list
            for i, issue in enumerate(data["issues"], 1):
                print(f"  {i}. {issue}")

            # Keyword placement detailsab
            # need to show the keyword placement details for each issue, if available
            # if placement := data.get("data", {}).get("keyword_placement", {}):
            #     print(f"\n  🔑 Keyword Placement:")
            #     for key, val in placement.items():
            #         if isinstance(val, dict):
            #             status = "✅" if val.get("present") else "❌"
            #             print(f"      {status} {key}: {val.get('value', 'N/A')}")
            #         else:
            #             status = "✅" if val else "❌"
            #             print(f"      {status} {key}")

            # Missing queries
            if missing := data.get("data", {}).get("missing_queries", []):
                print(f"\n  💡 Missing Queries ({len(missing)} relevant):")
                for q in missing:
                    if isinstance(q, dict):
                        if "score" in q:
                            print(f"      • {q['query']} (relevance: {q['score']:.2f})")
                        else:
                            clicks = q.get("total_clicks", 0)
                            impr = q.get("total_impressions", 0)
                            pos = q.get("avg_position", 0)
                            print(
                                f"      • {q.get('query', q)} (clicks: {clicks}, impr: {impr}, pos: {pos:.1f})"
                            )
                    else:
                        print(f"      • {q}")

            print()
    except Exception as e:
        print(f"Error printing report: {e}")


In [ ]:
# # | hide
# import dspy


# lm = dspy.LM(
#     model="openai/local",
#     api_base="http://localhost:8080/v1",
#     api_key="not-needed",
#     cache=False
    
    
# )
# print(lm("كيف الحال؟"))

# dspy.configure(lm=lm)


In [ ]:
from turtle import title
from seootter.sqlite_db import get_session

from tqdm.notebook import tqdm


with get_session() as session:
    articles = get_articles_by_website(session, 12)
    bar = tqdm(total=len(articles), desc="SEO Report")

    def progress(current, total, msg):
        bar.n = current
        bar.set_description(msg)
        bar.refresh()

    issues = generate_seo_report(
        session,
        12,
        "wrstat.com",
        is_quarto=True,
        progress_callback=progress,
        title_is_h1=True,
    )
    bar.close()
    print_issues_report(issues)


SEO Report: 0it [00:00, ?it/s]

SEO ANALYSIS REPORT
Pages analysed   : 0 / 0
Pages with issues: 0
Total issue count: 0
Duplicate titles : 0
Duplicate descs  : 0
